## Multi-organ dataset processing
Processes all organ datasets (bone marrow, bladder, liver, lung, kidney, heart, limb, brain) with metadata removal.

In [ ]:
import json
from pathlib import Path
from collections import Counter
import re

# ============================================================
# Configuration
# ============================================================
GENES_PREFIX = "Genes: "

# Define all organ patterns (brain has no prefix in filename)
ORGAN_PATTERNS = [
    ("bone-marrow", "bone-marrow_cell_data_part_*.json"),
    ("bladder", "bladder_cell_data_part_*.json"),
    ("liver", "liver_cell_data_part_*.json"),
    ("lung", "lung_cell_data_part_*.json"),
    ("kidney", "kidney_cell_data_part_*.json"),
    ("heart", "heart_cell_data_part_*.json"),
    ("limb-muscle", "limb-muscle_cell_data_part_*.json"),
    ("brain", "cell_data_part_*.json"),  # brain has no prefix
]

# Collect all source files from all organs
src_files = []
for organ_name, pattern in ORGAN_PATTERNS:
    files = sorted(Path("data").glob(pattern))
    # Filter out already-sorted outputs and classifier versions
    files = [p for p in files if "_sorted" not in p.stem and "_cls" not in p.stem]
    src_files.extend([(organ_name, p) for p in files])

print(f"found {len(src_files)} source files across all organs")
for organ_name, _ in ORGAN_PATTERNS:
    count = sum(1 for o, _ in src_files if o == organ_name)
    print(f"  {organ_name}: {count} files")

# Output folder
out_dir = Path("sorted_data")
out_dir.mkdir(parents=True, exist_ok=True)
print(f"\nwriting sorted outputs to: {out_dir.resolve()}")

# ============================================================
# Metadata removal function
# ============================================================
def strip_metadata(input_str):
    """
    Remove metadata lines like:
    \nGender: male\nClass: Kupffer cell\nTissue: Liver
    Returns the cleaned input string (only keeps the Genes: part).
    """
    if not isinstance(input_str, str):
        return input_str
    
    # Find where Genes: starts
    if GENES_PREFIX not in input_str:
        return input_str
    
    # Split by newline and keep only the Genes: line
    lines = input_str.split('\n')
    genes_lines = [line for line in lines if line.startswith(GENES_PREFIX)]
    
    if genes_lines:
        return genes_lines[0]  # Return just the Genes: line
    
    return input_str

# ============================================================
# Robust gene parser
# ============================================================
def parse_genes(input_str):
    """
    Safely parse:
    'Genes: gene1 count1 gene2 count2 ...'
    Returns list[(gene, count_str)]
    """
    if not isinstance(input_str, str):
        return []
    if not input_str.startswith(GENES_PREFIX):
        return []

    tokens = input_str[len(GENES_PREFIX):].split()
    pairs = []
    for i in range(0, len(tokens) - 1, 2):  # safe for odd token counts
        gene = tokens[i]
        count = tokens[i + 1]
        pairs.append((gene, count))
    return pairs

# ============================================================
# Step 1: Compute global gene frequency
# (frequency = number of times gene appears across all rows)
# ============================================================
gene_freq = Counter()
odd_token_rows = 0
total_rows = 0
bad_prefix_rows = 0

for organ_name, src in src_files:
    with src.open() as f:
        data = json.load(f)

    for row in data:
        total_rows += 1
        input_str = row.get("input", "")
        
        # Strip metadata before processing
        input_str = strip_metadata(input_str)

        if not (isinstance(input_str, str) and input_str.startswith(GENES_PREFIX)):
            bad_prefix_rows += 1
            continue

        tokens = input_str[len(GENES_PREFIX):].split()
        if len(tokens) % 2 != 0:
            odd_token_rows += 1

        genes = parse_genes(input_str)
        gene_freq.update(g for g, _ in genes)

print(f"\nunique genes: {len(gene_freq)}")
print(f"rows with odd gene token count: {odd_token_rows} / {total_rows}")
print(f"rows missing/invalid '{GENES_PREFIX}' prefix: {bad_prefix_rows} / {total_rows}")

# ============================================================
# Step 2: Rewrite datasets with frequency-sorted genes
# Output: sorted_data/{organ_name}__celldata_part_*_sorted_noorgan.json
# ============================================================
rows_with_no_genes = 0

for organ_name, src in src_files:
    with src.open() as f:
        data = json.load(f)

    sorted_rows = []

    for row in data:
        input_str = row.get("input", "")
        
        # Strip metadata before processing
        cleaned_input = strip_metadata(input_str)
        genes = parse_genes(cleaned_input)

        if not genes:
            rows_with_no_genes += 1
            # Keep row unchanged if genes can't be parsed
            sorted_rows.append(row)
            continue

        # Stable sort: by global frequency desc, then original position
        indexed = list(enumerate(genes))
        indexed.sort(key=lambda x: (-gene_freq[x[1][0]], x[0]))

        reordered = [genes[i] for i, _ in indexed]

        new_input = GENES_PREFIX + " ".join(f"{g} {c}" for g, c in reordered)

        new_row = dict(row)   # preserve all fields: instruction/output/label/etc.
        new_row["input"] = new_input
        sorted_rows.append(new_row)

    # Extract part number from original filename
    # e.g., bladder_cell_data_part_1.json -> part_1
    part_match = re.search(r'part_(\d+)', src.stem)
    part_num = part_match.group(1) if part_match else "unknown"
    
    # New filename format: {organ}__celldata_part_{num}_sorted_noorgan.json
    dest = out_dir / f"{organ_name}_celldata_part_{part_num}_sorted_noorgan.json"
    
    with dest.open("w") as f:
        json.dump(sorted_rows, f, ensure_ascii=True, indent=2)

    print(f"wrote {dest.name} ({len(sorted_rows)} rows)")

print(f"\nrows kept unchanged due to unparsable/empty genes: {rows_with_no_genes}")

found 88 source files across all organs
  bone-marrow: 11 files
  bladder: 11 files
  liver: 11 files
  lung: 11 files
  kidney: 11 files
  heart: 11 files
  limb-muscle: 11 files
  brain: 11 files

writing sorted outputs to: C:\Users\Hang Yu\Desktop\SCAP\Single_Cell_Aging_Prediction_LLaMA-Factory\sorted_data

unique genes: 22741
rows with odd gene token count: 0 / 94239
rows missing/invalid 'Genes: ' prefix: 0 / 94239
wrote bone-marrow__celldata_part_1_sorted_noorgan.json (2289 rows)
wrote bone-marrow__celldata_part_10_sorted_noorgan.json (2288 rows)
wrote bone-marrow__celldata_part_11_sorted_noorgan.json (2288 rows)
wrote bone-marrow__celldata_part_2_sorted_noorgan.json (2288 rows)
wrote bone-marrow__celldata_part_3_sorted_noorgan.json (2288 rows)
wrote bone-marrow__celldata_part_4_sorted_noorgan.json (2288 rows)
wrote bone-marrow__celldata_part_5_sorted_noorgan.json (2288 rows)
wrote bone-marrow__celldata_part_6_sorted_noorgan.json (2288 rows)
wrote bone-marrow__celldata_part_7_sort